In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [2]:
# ==========================================
# 1. DP-ENABLED CLIENT
# ==========================================
class DPFederatedClient:
    def __init__(self, client_id, X, y):
        self.client_id = client_id
        self.X = X
        self.y = y
        self.weights = None
        self.intercept = None

    def train_with_dp(self, global_weights, global_intercept, lr=0.01, 
                      clip_threshold=1.0, noise_multiplier=0.1):
        """
        Trains locally and applies Differential Privacy (Clipping + Noise).
        """
        # Sync with global model
        local_w = np.copy(global_weights)
        local_b = np.copy(global_intercept)
        
        # Simple Local Training (1 Epoch)
        preds = np.dot(self.X, local_w) + local_b
        error = preds - self.y
        dw = (1 / len(self.X)) * np.dot(self.X.T, error)
        db = (1 / len(self.X)) * np.sum(error)
        
        # --- DIFFERENTIAL PRIVACY STEPS ---
        
        # 1. Gradient Clipping: Limit the L2 norm of the update
        # This bounds the maximum influence a single client can have.
        total_norm = np.linalg.norm(dw)
        if total_norm > clip_threshold:
            dw = dw * (clip_threshold / total_norm)
        
        # 2. Add Gaussian Noise: Perturb the update
        # Noise scale is proportional to the clip threshold (sensitivity)
        noise_w = np.random.normal(0, clip_threshold * noise_multiplier, size=dw.shape)
        noise_b = np.random.normal(0, clip_threshold * noise_multiplier)
        
        noisy_dw = dw + noise_w
        noisy_db = db + noise_b
        
        # Apply the noisy updates locally
        self.weights = local_w - lr * noisy_dw
        self.intercept = local_b - lr * noisy_db
        
        return self.weights, self.intercept, len(self.X)

In [3]:
# ==========================================
# 2. AGGREGATING SERVER
# ==========================================
class DPServer:
    def __init__(self, input_dim):
        self.global_weights = np.random.randn(input_dim) * 0.01
        self.global_intercept = 0.0

    def aggregate(self, client_updates):
        total_n = sum(u[2] for u in client_updates)
        new_w = np.zeros_like(self.global_weights)
        new_b = 0.0
        
        for (w, b, n_k) in client_updates:
            weight_factor = n_k / total_n
            new_w += w * weight_factor
            new_b += b * weight_factor
            
        self.global_weights = new_w
        self.global_intercept = new_b

In [4]:
# ==========================================
# 3. SIMULATION
# ==========================================
def run_dp_federated_learning(rounds=20, noise_level=0.05):
    data = fetch_california_housing()
    X = StandardScaler().fit_transform(data.data)
    y = data.target
    
    # Partition data
    num_clients = 5
    X_shards = np.array_split(X, num_clients)
    y_shards = np.array_split(y, num_clients)
    
    server = DPServer(input_dim=X.shape[1])
    clients = [DPFederatedClient(i, X_shards[i], y_shards[i]) for i in range(num_clients)]
    
    print(f"Training with DP (Noise Multiplier: {noise_level})...")
    
    for r in range(rounds):
        updates = []
        for client in clients:
            # Clients add noise before sending updates to the server
            update = client.train_with_dp(server.global_weights, server.global_intercept, 
                                          noise_multiplier=noise_level)
            updates.append(update)
        
        server.aggregate(updates)
        
        if r % 5 == 0 or r == rounds - 1:
            preds = np.dot(X, server.global_weights) + server.global_intercept
            mse = mean_squared_error(y, preds)
            print(f"Round {r} | Global MSE: {mse:.4f}")

In [5]:
if __name__ == "__main__":
    # A noise_multiplier of 0.0 effectively turns off DP (for comparison)
    run_dp_federated_learning(rounds=20, noise_level=0.1)

Training with DP (Noise Multiplier: 0.1)...
Round 0 | Global MSE: 5.5165
Round 5 | Global MSE: 5.0774
Round 10 | Global MSE: 4.6837
Round 15 | Global MSE: 4.3254
Round 19 | Global MSE: 4.0596
